# GroupChat and Multi-Agent Teams

Notebook **08** scales from two-agent reflection (**05**) to **three-or-more specialist agents** in a **`RoundRobinGroupChat`**: **`docs_expert`**, **`db_expert`**, and **`security_expert`**. All agents **share context** via broadcast messages; you observe runs with **`team.run`** vs **`team.run_stream`**, and persist dialogue with **`save_state`** / **`load_state`**.

This notebook solves a **multi-part Notes API question** using the **c1–c5** corpus, diagrams the group chat topology, and compares the pattern to LangGraph supervisors (**02. LangGraph/11**).

**Prerequisites:** **05. Two-Agent Conversation Patterns**, **06. Tools and Function Registration**.

**What you'll learn:**

- How three specialists divide Notes API concerns
- How **shared context broadcast** keeps all agents aligned
- **`team.run`** vs **`run_stream`** for batch vs live observation
- **`save_state`** / **`load_state`** for team continuity (**12**)

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("autogen-agentchat: pip install autogen-agentchat autogen-ext[openai]")

try:
    import autogen_ext
    print("autogen-ext:", getattr(autogen_ext, "__version__", "installed"))
except ImportError:
    print("autogen-ext: pip install autogen-ext[openai]")

---

## 1. Why Multi-Agent Teams?

| Single agent (**06**) | Specialist team (**08**) |
|-----------------------|--------------------------|
| 10+ tools — routing confusion | 1–2 tools per expert |
| One bloated system prompt | Focused domain prompts |
| Hard to attribute errors | Trace shows which expert spoke |

Use teams when a single **`AssistantAgent`** consistently picks the wrong capability. Optimize the solo agent first (**06**), then split roles.

---

## 2. Group Chat Diagram

```
                         START (user task)
                                │
                                ▼
                 ┌──────────────────────────────┐
                 │     RoundRobinGroupChat        │
                 │  shared transcript (broadcast) │
                 └──────────────┬───────────────┘
                                │ round-robin turns
           ┌────────────────────┼────────────────────┐
           ▼                    ▼                    ▼
   ┌───────────────┐   ┌───────────────┐   ┌───────────────┐
   │  docs_expert  │   │   db_expert   │   │security_expert│
   │ FastAPI routes│   │ Alembic / DB  │   │ JWT / auth    │
   │  chunks c1,c5 │   │  chunks c2,c4 │   │  chunk c3     │
   └───────────────┘   └───────────────┘   └───────────────┘
           │                    │                    │
           └────────────────────┴────────────────────┘
                         loop until termination
```

Unlike **02. LangGraph/11**'s supervisor router, **round-robin** order is fixed — predictable but less flexible. See **09. GroupChatManager and Speaker Selection** for dynamic routing.

---

## 3. Notes API Corpus — Specialist Mapping

| Expert | Chunks | Domain |
|--------|--------|--------|
| **docs_expert** | c1, c5 | FastAPI routes, httpx testing |
| **db_expert** | c2, c4 | Alembic migrations, pytest fixtures |
| **security_expert** | c3 | JWT bearer authentication |

In [ ]:
NOTES = {
    "n1": "The Notes API is built with FastAPI. Routes live under /notes.",
    "n2": "Run alembic upgrade head after pulling database migrations.",
    "n3": "JWT bearer tokens use Authorization: Bearer header.",
    "n4": "Pytest fixtures for DB setup belong in conftest.py.",
}

DOC_CHUNKS = [
    {"id": "c1", "text": NOTES["n1"]},
    {"id": "c2", "text": NOTES["n2"]},
    {"id": "c3", "text": NOTES["n3"]},
    {"id": "c4", "text": NOTES["n4"]},
    {"id": "c5", "text": "Use httpx.AsyncClient in FastAPI tests for async endpoints."},
]

print("corpus:", [c["id"] for c in DOC_CHUNKS])

---

## 4. Async Specialist Tools

Each expert gets **narrow tools** — same registry pattern as **06**:

In [ ]:
import asyncio
from typing import Any


async def search_docs(query: str) -> str:
    """Keyword search over documentation chunks c1-c5."""
    await asyncio.sleep(0.01)
    tokens = set(query.lower().split())
    hits = [f"[{c['id']}] {c['text']}" for c in DOC_CHUNKS if tokens & set(c["text"].lower().split())]
    return "\n".join(hits) if hits else "No matching chunks."


async def get_chunk(chunk_id: str) -> str:
    """Fetch a documentation chunk by id (c1-c5)."""
    for c in DOC_CHUNKS:
        if c["id"] == chunk_id:
            return c["text"]
    return f"Unknown chunk: {chunk_id}"


TOOLS: dict[str, Any] = {"search_docs": search_docs, "get_chunk": get_chunk}
print("tools:", list(TOOLS))

---

## 5. Model Client

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0,
)
print("ready:", model_client.model)

---

## 6. Three Specialist Agents

In [ ]:
from autogen_agentchat.agents import AssistantAgent

docs_expert = AssistantAgent(
    name="docs_expert",
    model_client=model_client,
    tools=[search_docs, get_chunk],
    system_message=(
        "You are the FastAPI documentation expert for the Notes API. "
        "Focus on routes, httpx testing (c1, c5). Cite chunk ids. "
        "Be concise; defer DB topics to db_expert and auth to security_expert."
    ),
    reflect_on_tool_use=True,
)

db_expert = AssistantAgent(
    name="db_expert",
    model_client=model_client,
    tools=[search_docs, get_chunk],
    system_message=(
        "You are the database and testing expert. Focus on Alembic migrations (c2) "
        "and pytest fixtures in conftest.py (c4). Cite chunk ids."
    ),
    reflect_on_tool_use=True,
)

security_expert = AssistantAgent(
    name="security_expert",
    model_client=model_client,
    tools=[search_docs, get_chunk],
    system_message=(
        "You are the security expert. Focus on JWT bearer tokens (c3). "
        "Cite chunk ids. Warn about common auth mistakes."
    ),
    reflect_on_tool_use=True,
)

print("experts:", docs_expert.name, db_expert.name, security_expert.name)

---

## 7. `RoundRobinGroupChat` with 3+ Agents

Pass participants in speaking order. Each turn, the active agent sees **all prior messages** (shared context broadcast):

In [ ]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination

team_termination = MaxMessageTermination(max_messages=12) | TextMentionTermination("DONE")

notes_team = RoundRobinGroupChat(
    participants=[docs_expert, db_expert, security_expert],
    termination_condition=team_termination,
    max_turns=9,
)

print("turn order:", [a.name for a in notes_team._participants])

---

## 8. Shared Context Broadcast

In **`RoundRobinGroupChat`**, every agent message is appended to the **team transcript**. The next speaker sees:

1. The original user task
2. All prior expert responses
3. Tool summaries from any expert's tool calls

This is analogous to **`MessagesState`** with **`add_messages`** in LangGraph (**02. LangGraph/02**) — one growing conversation, not isolated per-agent memory.

---

## 9. `team.run` — Batch Result

**`run`** returns a single **`TaskResult`** when the team finishes:

In [ ]:
MULTI_PART_TASK = (
    "We are onboarding a new developer to the Notes API. Answer in three parts: "
    "(1) FastAPI route basics, (2) how to run Alembic migrations and where pytest "
    "fixtures live, (3) how JWT auth headers work. Each expert covers their domain. "
    "When all three parts are covered, the last expert should say DONE."
)

batch_result = await notes_team.run(task=MULTI_PART_TASK)
print("stop:", batch_result.stop_reason)
print("messages:", len(batch_result.messages))

---

## 10. Inspect Expert Contributions

In [ ]:
from autogen_agentchat.messages import TextMessage

for msg in batch_result.messages:
    if isinstance(msg, TextMessage) and msg.source in {"docs_expert", "db_expert", "security_expert"}:
        print(f"--- {msg.source} ---")
        print((msg.content or "")[:250])
        print()

---

## 11. `team.run_stream` — Live Observation

**`run_stream`** yields messages as they are produced — ideal for notebooks and **`Console`** (**15**):

In [ ]:
from autogen_agentchat.base import TaskResult

await notes_team.reset()

async for event in notes_team.run_stream(task=MULTI_PART_TASK):
    if isinstance(event, TaskResult):
        print("\n=== TEAM STOP ===", event.stop_reason)
    elif isinstance(event, TextMessage):
        print(f"[{event.source}]", (event.content or "")[:120].replace("\n", " "))

---

## 12. Team State — `save_state` / `load_state`

Persist team context for resume (**12. Memory and Conversation State**):

In [ ]:
state = await notes_team.save_state()
print("state keys:", list(state.keys())[:5], "...")

await notes_team.reset()
print("after reset, message count from last run cleared")

await notes_team.load_state(state)
print("state restored — team can continue with run/run_stream")

---

## 13. Follow-Up Task on Restored Team

After **`load_state`**, call **`run`** with a follow-up — the team retains prior context:

In [ ]:
# follow_up = await notes_team.run(task="db_expert: which chunk mentions conftest.py?")
print("Uncomment follow_up run after load_state to continue the same session.")

---

## 14. `max_turns` with Three Agents

With 3 experts, each "round" consumes **3 turns**. Set **`max_turns`** ≥ 3 × expected rounds:

| `max_turns` | Effect with 3 agents |
|-------------|---------------------|
| 3 | One full round only |
| 6 | Two rounds |
| 9 | Three rounds (default teaching setting) |

In [ ]:
print("max_turns:", notes_team._max_turns)
print("experts per round:", len(notes_team._participants))

---

## 15. Compare to LangGraph Supervisor (**02. LangGraph/11**)

| Aspect | AutoGen `RoundRobinGroupChat` | LangGraph supervisor |
|--------|------------------------------|----------------------|
| Routing | Fixed turn order | Model/router picks next worker |
| Context | Shared team transcript | `MessagesState` + handoffs |
| Termination | `TerminationCondition` | Conditional edges to END |
| Streaming | `run_stream` | `astream` / `stream` |
| State | `save_state` / `load_state` | Checkpointers (**02. LangGraph/08**) |

Round-robin is simpler to reason about; supervisors scale better when turn order should be dynamic (**09**).

---

## 16. Design Guidelines

| Guideline | Rationale |
|-----------|-----------|
| **1–2 tools per expert** | Keeps routing reliable |
| **Cross-defer in system prompts** | Experts acknowledge boundaries |
| **Shared termination keyword** | `DONE` or `APPROVE` ends cleanly |
| **`max_turns` = rounds × agents** | Budget for full cycles |
| **`reset` between unrelated tasks** | Avoid context pollution |
| **`save_state` before long pauses** | Resume without re-running (**12**) |

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Too many experts | Shallow turns, high cost | Merge related roles |
| Identical system prompts | Redundant answers | Specialize per domain |
| No termination condition | Runs until `max_turns` only | Add `TextMentionTermination` |
| `max_turns` not divisible by N | Last agent cut off mid-round | Round up turns |
| Expecting dynamic routing | Wrong expert speaks | Use **09** SelectorGroupChat |
| Skipping `reset` between demos | Prior task bleeds in | `await team.reset()` |

---

## 18. Summary

**`RoundRobinGroupChat`** with **three specialists** solves multi-part Notes API questions: each expert contributes from the **c1–c5** corpus, broadcasting into a **shared transcript**. Use **`team.run`** for batch results, **`run_stream`** for live traces, and **`save_state`** / **`load_state`** for continuity.

Key takeaways:

- **Shared context broadcast** keeps all experts aligned on the user task.
- **`max_turns`** should account for three agents per round.
- For dynamic speaker selection, graduate to **09. GroupChatManager and Speaker Selection**.

Demonstrations built docs/db/security experts, ran a multi-part onboarding task, streamed team output, and saved/restored team state.

Next: **09. GroupChatManager and Speaker Selection** — when round-robin order is too rigid.